# Particle Yield Dilation Validation

This notebook validates the critical-bitmap particle yield calculator against a direct Monte Carlo simulation.

The toy setup uses a single D2W interface with 30% critical signal bumps and no redundant bumps. The calculator uses the distance-transform form of bitmap dilation, while the Monte Carlo model samples particles, generates main voids, and checks whether each void overlaps any critical bump.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

ROOT = Path.cwd()
D2W_DIR = ROOT / "D2W" if (ROOT / "D2W").is_dir() else ROOT
if str(D2W_DIR) not in sys.path:
    sys.path.insert(0, str(D2W_DIR))

from defect_yield_calculator import interface_particle_yield_from_critical_bitmap

In [ ]:
PAD_ARR_ROW = 100
PAD_ARR_COL = 100
PITCH_UM = 40.0
PAD_ARR_W_UM = (PAD_ARR_COL - 1) * PITCH_UM
PAD_ARR_L_UM = (PAD_ARR_ROW - 1) * PITCH_UM

cfg = SimpleNamespace(
    PITCH_r_um=PITCH_UM,
    PITCH_c_um=PITCH_UM,
    DIE_W_um=4200.0,
    DIE_L_um=4200.0,
    PAD_ARR_ROW=PAD_ARR_ROW,
    PAD_ARR_COL=PAD_ARR_COL,
    PAD_ARR_W_um=PAD_ARR_W_UM,
    PAD_ARR_L_um=PAD_ARR_L_UM,
    PAD_TOP_R_um=5.0,
    D0=1.0e-9,
    D1=1.0e-8,
    EDGE_REGION_WIDTH_um=300.0,
    first_contact="center",
    k_r=1.8e-4,
    k_r0=230.0,
    t_0=0.1,
    z=3.0,
)

CRITICAL_RATIO = 0.30
N_MC_TRIALS = 50_000
RNG_SEED = 42


In [ ]:
def pad_coordinate_mesh(cfg):
    rows = np.arange(cfg.PAD_ARR_ROW)
    cols = np.arange(cfg.PAD_ARR_COL)
    col_grid, row_grid = np.meshgrid(cols, rows, indexing="xy")
    x_um = -cfg.PAD_ARR_W_um / 2.0 + col_grid * cfg.PITCH_c_um
    y_um = cfg.PAD_ARR_L_um / 2.0 - row_grid * cfg.PITCH_r_um
    return x_um, y_um


def make_critical_bitmap(cfg, pattern, ratio=0.30, seed=1):
    x_um, y_um = pad_coordinate_mesh(cfg)
    num_pads = cfg.PAD_ARR_ROW * cfg.PAD_ARR_COL
    num_critical = int(round(ratio * num_pads))
    bitmap = np.zeros(num_pads, dtype=bool)

    if pattern == "random":
        rng = np.random.default_rng(seed)
        selected = rng.choice(num_pads, size=num_critical, replace=False)
    elif pattern == "center":
        metric = np.sqrt(x_um**2 + y_um**2).ravel()
        selected = np.argsort(metric)[:num_critical]
    elif pattern == "edge":
        dist_to_edge = np.minimum(
            cfg.DIE_W_um / 2.0 - np.abs(x_um),
            cfg.DIE_L_um / 2.0 - np.abs(y_um),
        ).ravel()
        selected = np.argsort(dist_to_edge)[:num_critical]
    else:
        raise ValueError(f"Unknown pattern: {pattern}")

    bitmap[selected] = True
    return bitmap.reshape(cfg.PAD_ARR_ROW, cfg.PAD_ARR_COL)


patterns = ["random", "center", "edge"]
critical_bitmaps = {pattern: make_critical_bitmap(cfg, pattern, CRITICAL_RATIO) for pattern in patterns}

fig, axes = plt.subplots(1, len(patterns), figsize=(10, 3), constrained_layout=True)
for ax, pattern in zip(axes, patterns):
    ax.imshow(critical_bitmaps[pattern], cmap="gray_r", interpolation="nearest")
    ax.set_title(pattern)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

In [ ]:
def distance_to_first_contact(cfg, x_um, y_um):
    if cfg.first_contact == "center":
        return np.sqrt(x_um**2 + y_um**2)
    if cfg.first_contact == "vertical-edge":
        return np.abs(cfg.DIE_W_um / 2.0 + x_um)
    if cfg.first_contact == "horizontal-edge":
        return np.abs(cfg.DIE_L_um / 2.0 + y_um)
    if cfg.first_contact == "corner":
        return np.sqrt((cfg.DIE_W_um / 2.0 + x_um) ** 2 + (cfg.DIE_L_um / 2.0 + y_um) ** 2)
    raise ValueError(f"Unsupported first_contact mode: {cfg.first_contact}")


def sample_particle_centers_with_edge_density(cfg, n_particles, rng):
    """Sample particle centers from the D0 + edge-excess density used by the calculator."""
    if n_particles == 0:
        return np.empty(0), np.empty(0)

    D0_area = cfg.D0 * cfg.DIE_W_um * cfg.DIE_L_um
    edge_width = min(cfg.EDGE_REGION_WIDTH_um, cfg.DIE_W_um / 2.0, cfg.DIE_L_um / 2.0)
    delta_D = max(cfg.D1 - cfg.D0, 0.0)
    edge_excess_area = 0.0
    if delta_D > 0.0 and edge_width > 0.0:
        # Integral of max(0, 1 - distance_to_nearest_edge / w) over a rectangle.
        edge_excess_area = delta_D * (
            edge_width * (cfg.DIE_W_um + cfg.DIE_L_um)
            - (4.0 / 3.0) * edge_width**2
        )

    use_edge = rng.random(n_particles) < (edge_excess_area / (D0_area + edge_excess_area) if (D0_area + edge_excess_area) > 0 else 0.0)
    x_um = rng.uniform(-cfg.DIE_W_um / 2.0, cfg.DIE_W_um / 2.0, n_particles)
    y_um = rng.uniform(-cfg.DIE_L_um / 2.0, cfg.DIE_L_um / 2.0, n_particles)

    n_edge = int(np.count_nonzero(use_edge))
    if n_edge > 0:
        accepted_x = []
        accepted_y = []
        while len(accepted_x) < n_edge:
            batch = max(1024, 2 * (n_edge - len(accepted_x)))
            x_try = rng.uniform(-cfg.DIE_W_um / 2.0, cfg.DIE_W_um / 2.0, batch)
            y_try = rng.uniform(-cfg.DIE_L_um / 2.0, cfg.DIE_L_um / 2.0, batch)
            dist_to_edge = np.minimum(cfg.DIE_W_um / 2.0 - np.abs(x_try), cfg.DIE_L_um / 2.0 - np.abs(y_try))
            keep_prob = np.clip(1.0 - dist_to_edge / edge_width, 0.0, 1.0)
            keep = rng.random(batch) < keep_prob
            accepted_x.extend(x_try[keep].tolist())
            accepted_y.extend(y_try[keep].tolist())
        x_um[use_edge] = np.asarray(accepted_x[:n_edge])
        y_um[use_edge] = np.asarray(accepted_y[:n_edge])

    return x_um, y_um


def monte_carlo_particle_yield(cfg, critical_bitmap, n_trials=50_000, seed=42):
    rng = np.random.default_rng(seed)
    uniform_mean = cfg.D0 * cfg.DIE_W_um * cfg.DIE_L_um
    edge_width = min(cfg.EDGE_REGION_WIDTH_um, cfg.DIE_W_um / 2.0, cfg.DIE_L_um / 2.0)
    delta_D = max(cfg.D1 - cfg.D0, 0.0)
    edge_excess_mean = 0.0
    if delta_D > 0.0 and edge_width > 0.0:
        edge_excess_mean = delta_D * (
            edge_width * (cfg.DIE_W_um + cfg.DIE_L_um)
            - (4.0 / 3.0) * edge_width**2
        )
    mean_particles_per_trial = uniform_mean + edge_excess_mean

    particle_counts = rng.poisson(mean_particles_per_trial, size=n_trials)
    total_particles = int(particle_counts.sum())
    if total_particles == 0:
        return 1.0, 0.0, 0

    trial_ids = np.repeat(np.arange(n_trials), particle_counts)
    x_um, y_um = sample_particle_centers_with_edge_density(cfg, total_particles, rng)

    u = rng.random(total_particles)
    particle_t_um = cfg.t_0 / (1.0 - u) ** (1.0 / (cfg.z - 1.0))
    r_mv_um = (cfg.k_r * distance_to_first_contact(cfg, x_um, y_um) + cfg.k_r0) * np.sqrt(particle_t_um)

    pad_x_um, pad_y_um = pad_coordinate_mesh(cfg)
    critical_coords_um = np.column_stack([pad_x_um[critical_bitmap], pad_y_um[critical_bitmap]])
    nearest_critical_dist_um, _ = cKDTree(critical_coords_um).query(np.column_stack([x_um, y_um]), k=1)
    fatal_particle = nearest_critical_dist_um <= (r_mv_um + cfg.PAD_TOP_R_um)

    failed_trial = np.zeros(n_trials, dtype=bool)
    failed_trial[trial_ids[fatal_particle]] = True
    mc_yield = float(np.mean(~failed_trial))
    mc_standard_error = float(np.sqrt(mc_yield * (1.0 - mc_yield) / n_trials))
    return mc_yield, mc_standard_error, total_particles


In [ ]:
records = []
for pattern, critical_bitmap in critical_bitmaps.items():
    calc_t0 = time.perf_counter()
    calc_yield, calc_info = interface_particle_yield_from_critical_bitmap(
        cfg,
        {"CRITICAL_PAD_BITMAP": critical_bitmap},
    )
    calc_time_s = time.perf_counter() - calc_t0

    mc_t0 = time.perf_counter()
    mc_yield, mc_se, total_particles = monte_carlo_particle_yield(
        cfg,
        critical_bitmap,
        n_trials=N_MC_TRIALS,
        seed=RNG_SEED,
    )
    mc_time_s = time.perf_counter() - mc_t0

    records.append(
        {
            "pattern": pattern,
            "calc_yield": calc_yield,
            "mc_yield": mc_yield,
            "abs_error": abs(calc_yield - mc_yield),
            "mc_standard_error": mc_se,
            "calc_time_s": calc_time_s,
            "mc_time_s": mc_time_s,
            "mc_particles": total_particles,
            "avg_fatal_particles_calc": calc_info["avg_fatal_particles"],
            "effective_critical_area_um2": calc_info["effective_critical_area_um2"],
        }
    )

results_df = pd.DataFrame(records)
results_df

In [ ]:
x = np.arange(len(results_df))
width = 0.36

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.bar(x - width / 2, results_df["calc_yield"], width, label="calculator")
ax.bar(x + width / 2, results_df["mc_yield"], width, label="Monte Carlo")
ax.errorbar(
    x + width / 2,
    results_df["mc_yield"],
    yerr=2.0 * results_df["mc_standard_error"],
    fmt="none",
    color="black",
    capsize=3,
    label="Monte Carlo 2 SE",
)
ax.set_xticks(x)
ax.set_xticklabels(results_df["pattern"])
ax.set_ylabel("particle yield")
ax.set_ylim(0.0, 1.02)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.bar(results_df["pattern"], results_df["calc_time_s"], label="calculator")
ax.bar(results_df["pattern"], results_df["mc_time_s"], bottom=results_df["calc_time_s"], label="Monte Carlo")
ax.set_ylabel("runtime (s)")
ax.legend()
plt.show()

## Notes

- This validation uses a 100x100 pad array with `D1 = 10 * D0`.
- The calculator is deterministic and runs on a grid whose default pixel size is the pad pitch.
- The Monte Carlo result has sampling noise, reported as `mc_standard_error`.
- Any systematic difference beyond the Monte Carlo error is mainly from the finite grid approximation in the bitmap/distance-transform critical area calculation.
- The Monte Carlo sampler uses the same linear edge-enhanced particle-density profile as the calculator.


## 10000 x 10000 Benchmark

A standalone run on the current workstation used a 10000 x 10000 critical bitmap, 1 um pitch, 30% critical signal bumps, `D1 = 10 * D0`, and `DEFECT_CALC_CHUNK_ROWS = 256`.

Recorded result:

```text
critical ratio = 0.300
calculator time = 19.634 s
process wall time = 20.74 s
peak RSS = 3,515,060 KB = 3.35 GiB
grid shape = (10002, 10002)
yield = 0.98482433
```

The cell below can be enabled to rerun this benchmark. It allocates several GiB of memory, so leave `RUN_10000_BENCHMARK = False` unless the machine has enough RAM.


In [ ]:
RUN_10000_BENCHMARK = False

if RUN_10000_BENCHMARK:
    import resource

    n = 10_000
    pitch_um = 1.0
    large_cfg = SimpleNamespace(
        PITCH_r_um=pitch_um,
        PITCH_c_um=pitch_um,
        DIE_W_um=n * pitch_um,
        DIE_L_um=n * pitch_um,
        PAD_ARR_ROW=n,
        PAD_ARR_COL=n,
        PAD_ARR_W_um=(n - 1) * pitch_um,
        PAD_ARR_L_um=(n - 1) * pitch_um,
        PAD_TOP_R_um=0.2,
        D0=1.0e-10,
        D1=1.0e-9,
        EDGE_REGION_WIDTH_um=300.0,
        first_contact="center",
        k_r=1.8e-4,
        k_r0=230.0,
        t_0=0.1,
        z=3.0,
        DEFECT_CALC_CHUNK_ROWS=256,
    )

    t0 = time.perf_counter()
    large_critical = np.zeros((n, n), dtype=bool)
    large_critical[:, 0::10] = True
    large_critical[:, 1::10] = True
    large_critical[:, 2::10] = True
    build_time_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    large_yield, large_info = interface_particle_yield_from_critical_bitmap(
        large_cfg,
        {"CRITICAL_PAD_BITMAP": large_critical},
    )
    calc_time_s = time.perf_counter() - t0
    rss_gib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2

    print(f"critical ratio: {large_critical.mean():.3f}")
    print(f"bitmap build time: {build_time_s:.3f} s")
    print(f"calculator time: {calc_time_s:.3f} s")
    print(f"peak RSS: {rss_gib:.2f} GiB")
    print(f"yield: {large_yield:.8f}")
    print(large_info)
